# Frozen DeepSets Fingerprint Baselines\n
\n
Train the same DeepSets-style fingerprint head on pre-computed frozen SSL embeddings (no backbone forward, no backbone gradients).\n
\n
Conditions trained in BCE-first order:\n
1. morgan_2048 + bce_logits\n
2. maccs_166 + bce_logits\n
3. map4_2048 + bce_logits\n
4. morgan_2048 + cos\n
5. maccs_166 + cos\n
6. map4_2048 + cos\n

In [22]:
from __future__ import annotations

import copy
import json
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Paths
FINETUNING_HDF5 = PROJECT_ROOT / 'dreams-thesis-wa/data/processed/MassSpecGym_splits/finetuning.hdf5'
FINGERPRINT_CACHE_NPZ = PROJECT_ROOT / 'dreams-thesis-wa/data/processed/MassSpecGym_splits/fingerprint_cache.npz'

# Expected source of precomputed frozen SSL embeddings in HDF5 format (N, 1024).
# You can point this to your external file and the key auto-detection below will try common names.
EMBEDDING_HDF5 = PROJECT_ROOT / 'dreams-thesis-wa/data/processed/MassSpecGym_splits/finetuning_with_ssl_embeddings.hdf5'
EMBEDDING_KEYS_CANDIDATES = [
    'ssl_embedding',
    'ssl_embeddings',
    'embedding',
    'embeddings',
    'precursor_embedding',
    'precursor_embeddings',
]

OUTPUT_ROOT = PROJECT_ROOT / 'dreams-thesis-wa/results/frozen_deepsets_baselines'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Hyperparameters mirrored from fine_tune_test.sh where applicable (minus backbone).
SEED = 3407
LR = 1.5e-5
WEIGHT_DECAY = 0.0
BATCH_SIZE = 256
MAX_EPOCHS = 103
EARLY_STOPPING_PATIENCE = 20
DROPOUT = 0.0
NUM_WORKERS = 0
PIN_MEMORY = False
VAL_CHECK_INTERVAL = 0.5 


# Force CPU first to rule out MPS numerical instability.
DEVICE = torch.device('cpu')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DEVICE: {DEVICE}')
print(f'Output dir: {OUTPUT_ROOT}')

PROJECT_ROOT: /Users/wouterachterberg/coding/DreaMS
DEVICE: cpu
Output dir: /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/frozen_deepsets_baselines


In [23]:
def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def decode_utf8_col(values: np.ndarray) -> np.ndarray:
    if len(values) == 0:
        return values.astype(str)
    if isinstance(values[0], (bytes, np.bytes_)):
        return np.array([v.decode('utf-8') for v in values], dtype=object)
    return values.astype(object, copy=False)


def find_embedding_key(h5f: h5py.File, candidates: list[str]) -> str:
    for k in candidates:
        if k in h5f and len(h5f[k].shape) == 2 and h5f[k].shape[1] == 1024:
            return k

    # Fallback: detect any (N, 1024) matrix.
    for k in h5f.keys():
        ds = h5f[k]
        if hasattr(ds, 'shape') and len(ds.shape) == 2 and ds.shape[1] == 1024:
            return k

    raise KeyError(
        'Could not find a (N,1024) embedding dataset in embedding HDF5. '
        f'Available keys: {sorted(list(h5f.keys()))}'
    )


def load_frozen_embeddings_from_hdf5(path: Path) -> np.ndarray:
    if not path.exists():
        raise FileNotFoundError(
            f'Embedding HDF5 not found at {path}. '
            'Set EMBEDDING_HDF5 to your precomputed frozen embedding file.'
        )

    with h5py.File(path, 'r') as f:
        emb_key = find_embedding_key(f, EMBEDDING_KEYS_CANDIDATES)
        embs = f[emb_key][:].astype(np.float32)
        print(f'Using embedding key: {emb_key} from {path}')
        print(f'Embeddings shape: {embs.shape}, dtype={embs.dtype}')
    return embs


def load_fold_vector(path: Path) -> np.ndarray:
    with h5py.File(path, 'r') as f:
        fold = decode_utf8_col(f['fold'][:])
    return np.array(fold, dtype=object)


def load_fingerprint_targets(path: Path) -> dict[str, np.ndarray]:
    if not path.exists():
        raise FileNotFoundError(f'Fingerprint cache missing: {path}')
    z = np.load(path)
    mapping = {
        'morgan_2048': z['morgan_fps'].astype(np.float32),
        'maccs_166': z['maccs_fps'].astype(np.float32),
        'map4_2048': z['map4_fps'].astype(np.float32),
    }
    return mapping


class EmbFpDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray):
        self.x = torch.from_numpy(x.astype(np.float32, copy=False))
        self.y = torch.from_numpy(y.astype(np.float32, copy=False))

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


class CosSimLoss(nn.Module):
    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return 1 - F.cosine_similarity(inputs, targets).mean()


class FrozenEmbeddingDeepSetsHead(nn.Module):
    """
    DeepSets-style head on frozen embeddings:
    phi: 1024 -> 1024 + dropout, sum-pool over set dimension, rho: 1024 -> n_bits.

    Since each sample is a single 1024-d embedding vector, we represent it as a set with one element.
    """

    def __init__(self, out_dim: int, dropout: float = 0.0):
        super().__init__()
        self.phi = nn.Sequential(
            nn.Linear(1024, 1024, bias=False),
            nn.Dropout(dropout),
        )
        self.rho = nn.Linear(1024, out_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, 1024] -> [B, 1, 1024] to mimic a single-element set.
        x = x.unsqueeze(1)
        x = self.phi(x)
        x = x.sum(dim=1)
        return self.rho(x)


@dataclass
class RunConfig:
    fp_kind: str
    loss_kind: str
    run_tag: str
    pos_weight: Optional[float] = None


RUNS = [
    # BCE first (priority)
    RunConfig('morgan_2048', 'bce_logits', 'frozen_morgan_2048_bce'),
    RunConfig('maccs_166', 'bce_logits', 'frozen_maccs_166_bce'),
    RunConfig('map4_2048', 'bce_logits', 'frozen_map4_2048_bce'),
    # Cosine objective
    RunConfig('morgan_2048', 'cos', 'frozen_morgan_2048_cos'),
    RunConfig('maccs_166', 'cos', 'frozen_maccs_166_cos'),
    RunConfig('map4_2048', 'cos', 'frozen_map4_2048_cos'),
]

In [24]:
set_seed(SEED)

emb_all = load_frozen_embeddings_from_hdf5(EMBEDDING_HDF5)
fold = load_fold_vector(FINETUNING_HDF5)
fps = load_fingerprint_targets(FINGERPRINT_CACHE_NPZ)

if len(emb_all) != len(fold):
    raise ValueError(f'Length mismatch: embeddings={len(emb_all)} vs fold={len(fold)}')

for name, arr in fps.items():
    if len(arr) != len(fold):
        raise ValueError(f'Length mismatch for {name}: {len(arr)} vs fold={len(fold)}')

train_mask = fold == 'train'
val_mask = fold == 'val'
if val_mask.sum() == 0:
    # Fallback in case split labels differ.
    val_mask = fold == 'test'

print('Data summary:')
print(f'  Total: {len(fold):,}')
print(f'  Train: {int(train_mask.sum()):,}')
print(f'  Val:   {int(val_mask.sum()):,}')
print(f'  Embedding dim: {emb_all.shape[1]}')

Using embedding key: ssl_embedding from /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/data/processed/MassSpecGym_splits/finetuning_with_ssl_embeddings.hdf5
Embeddings shape: (159271, 1024), dtype=float32
Data summary:
  Total: 159,271
  Train: 135,856
  Val:   23,415
  Embedding dim: 1024


In [25]:


def make_loss(loss_kind: str, n_bits: int, pos_weight: Optional[float]) -> nn.Module:
    if loss_kind == 'cos':
        return CosSimLoss()
    if loss_kind == 'bce_logits':
        if pos_weight is None:
            return nn.BCEWithLogitsLoss()
        pw = torch.full((n_bits,), float(pos_weight), dtype=torch.float32, device=DEVICE)
        return nn.BCEWithLogitsLoss(pos_weight=pw)
    raise ValueError(f'Unsupported loss kind: {loss_kind}')


def eval_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, loss_kind: str) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            pred = model(xb)
            pred_for_loss = pred
            if loss_kind in {'bce', 'cross_entropy'}:
                pred_for_loss = torch.sigmoid(pred)
            loss = criterion(pred_for_loss, yb)
            losses.append(loss.detach().cpu().item())
    return float(np.mean(losses))


def train_one_condition(cfg: RunConfig) -> dict:
    run_dir = OUTPUT_ROOT / cfg.run_tag
    ckpt_dir = run_dir / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    y_all = fps[cfg.fp_kind]
    x_train, y_train = emb_all[train_mask], y_all[train_mask]
    x_val, y_val = emb_all[val_mask], y_all[val_mask]

    train_ds = EmbFpDataset(x_train, y_train)
    val_ds = EmbFpDataset(x_val, y_val)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
    )

    model = FrozenEmbeddingDeepSetsHead(out_dim=y_train.shape[1], dropout=DROPOUT).to(DEVICE)
    criterion = make_loss(cfg.loss_kind, y_train.shape[1], cfg.pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_val = float('inf')
    best_epoch = -1
    best_state = None
    no_improve = 0
    history = []

    num_train_batches = len(train_loader)
    mid_epoch_step = max(1, int(np.ceil(num_train_batches * VAL_CHECK_INTERVAL)))

    t0 = time.perf_counter()
    last_epoch = 0
    early_stop_triggered = False
    val_check_count = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_losses = []

        for step, (xb, yb) in enumerate(train_loader, start=1):
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            pred_for_loss = pred
            if cfg.loss_kind in {'bce', 'cross_entropy'}:
                pred_for_loss = torch.sigmoid(pred)
            loss = criterion(pred_for_loss, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.detach().cpu().item())

            should_validate = (step == mid_epoch_step) or (step == num_train_batches)
            if not should_validate:
                continue

            train_loss_so_far = float(np.mean(train_losses))
            val_loss = eval_epoch(model, val_loader, criterion, cfg.loss_kind)
            val_check_count += 1

            history.append({
                'epoch': epoch,
                'val_check': val_check_count,
                'step_in_epoch': step,
                'train_loss': train_loss_so_far,
                'val_loss': val_loss,
            })

            improved = val_loss < best_val
            if improved:
                best_val = val_loss
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                torch.save(
                    {
                        'epoch': epoch,
                        'step_in_epoch': step,
                        'val_check': val_check_count,
                        'val_loss': val_loss,
                        'model_state_dict': best_state,
                        'optimizer_state_dict': optimizer.state_dict(),
                        'config': cfg.__dict__,
                        'n_bits': int(y_train.shape[1]),
                    },
                    ckpt_dir / 'best.ckpt',
                )
                no_improve = 0
            else:
                no_improve += 1

            print(
                f"[{cfg.run_tag}] epoch={epoch:03d} check={val_check_count:03d} "
                f"step={step:04d}/{num_train_batches:04d} "
                f"train={train_loss_so_far:.6f} val={val_loss:.6f}"
                + ('  <-- best' if improved else '')
            )

            if no_improve >= EARLY_STOPPING_PATIENCE:
                print(
                    f'[{cfg.run_tag}] Early stopping at epoch {epoch}, val_check {val_check_count} '
                    f'(patience={EARLY_STOPPING_PATIENCE}, val_check_interval={VAL_CHECK_INTERVAL})'
                )
                early_stop_triggered = True
                break

        last_epoch = epoch
        if early_stop_triggered:
            break

    seconds = time.perf_counter() - t0

    # Store compact run metadata and history for later comparison.
    metadata = {
        'run_tag': cfg.run_tag,
        'fp_kind': cfg.fp_kind,
        'loss_kind': cfg.loss_kind,
        'pos_weight': cfg.pos_weight,
        'n_train': int(train_mask.sum()),
        'n_val': int(val_mask.sum()),
        'best_epoch': best_epoch,
        'best_val_loss': float(best_val),
        'epochs_ran': int(last_epoch),
        'val_checks_ran': int(val_check_count),
        'val_check_interval': float(VAL_CHECK_INTERVAL),
        'seconds_total': float(seconds),
        'lr': LR,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'dropout': DROPOUT,
    }

    (run_dir / 'history.json').write_text(json.dumps(history, indent=2))
    (run_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2))

    return metadata

In [26]:
all_results = []

for cfg in RUNS:
    print('\n' + '=' * 90)
    print(f'Training frozen baseline: {cfg.run_tag}')
    print('=' * 90)
    result = train_one_condition(cfg)
    all_results.append(result)

results_df = pd.DataFrame(all_results).sort_values(['loss_kind', 'fp_kind']).reset_index(drop=True)
results_df


Training frozen baseline: frozen_morgan_2048_bce
[frozen_morgan_2048_bce] epoch=001 check=001 step=0266/0531 train=0.290869 val=0.094111  <-- best
[frozen_morgan_2048_bce] epoch=001 check=002 step=0531/0531 train=0.188934 val=0.080618  <-- best
[frozen_morgan_2048_bce] epoch=002 check=003 step=0266/0531 train=0.079985 val=0.076404  <-- best
[frozen_morgan_2048_bce] epoch=002 check=004 step=0531/0531 train=0.078369 val=0.074159  <-- best
[frozen_morgan_2048_bce] epoch=003 check=005 step=0266/0531 train=0.074785 val=0.072653  <-- best
[frozen_morgan_2048_bce] epoch=003 check=006 step=0531/0531 train=0.074093 val=0.071527  <-- best
[frozen_morgan_2048_bce] epoch=004 check=007 step=0266/0531 train=0.072214 val=0.070618  <-- best
[frozen_morgan_2048_bce] epoch=004 check=008 step=0531/0531 train=0.071554 val=0.069866  <-- best
[frozen_morgan_2048_bce] epoch=005 check=009 step=0266/0531 train=0.070148 val=0.069217  <-- best
[frozen_morgan_2048_bce] epoch=005 check=010 step=0531/0531 train=0.

KeyboardInterrupt: 

In [ ]:
# Save a run-level summary table.
summary_csv = OUTPUT_ROOT / 'frozen_baseline_summary.csv'
results_df.to_csv(summary_csv, index=False)
print(f'Saved summary: {summary_csv}')
display(results_df[['run_tag', 'fp_kind', 'loss_kind', 'best_val_loss', 'best_epoch', 'seconds_total']])

Saved summary: /Users/wouterachterberg/coding/DreaMS/dreams-thesis-wa/results/frozen_deepsets_baselines/frozen_baseline_summary.csv


,run_tag,fp_kind,loss_kind,best_val_loss,best_epoch,seconds_total
0,frozen_maccs_166_bce,maccs_166,bce_logits,0.282593,24,94.529502
1,frozen_map4_2048_bce,map4_2048,bce_logits,0.475535,21,167.716745
2,frozen_morgan_2048_bce,morgan_2048,bce_logits,0.063586,32,233.102886
3,frozen_maccs_166_cos,maccs_166,cos,0.198298,34,136.247667
4,frozen_map4_2048_cos,map4_2048,cos,0.461236,22,198.955269
5,frozen_morgan_2048_cos,morgan_2048,cos,0.452134,37,260.106913


In [27]:
# Comparability-aware summary for BCE results.
# Raw BCE val loss is NOT comparable across fingerprint families with different bit-count/density.

bce_df = results_df[results_df['loss_kind'] == 'bce_logits'].copy()
if bce_df.empty:
    print('No BCE runs found in results_df.')
else:
    rows = []
    for _, r in bce_df.iterrows():
        fp_kind = r['fp_kind']
        y_train = fps[fp_kind][train_mask]
        rows.append(
            {
                'run_tag': r['run_tag'],
                'fp_kind': fp_kind,
                'n_bits': int(y_train.shape[1]),
                'train_bit_density': float(y_train.mean()),
                'best_val_loss': float(r['best_val_loss']),
                'best_epoch': int(r['best_epoch']),
            }
        )

    bce_context_df = pd.DataFrame(rows).sort_values('fp_kind').reset_index(drop=True)
    display(bce_context_df)

    print('Interpretation:')
    print('- Do NOT compare raw BCE losses across different fingerprint types.')
    print('- Different n_bits and bit densities change baseline BCE scale.')
    print('- Use val loss only for within-type comparisons (e.g., Morgan frozen vs Morgan fine-tuned).')
    print('- For cross-type comparison, use evaluation metrics: per-bit AUROC, Tanimoto@optimal-threshold, retrieval acc@k.')

,run_tag,fp_kind,n_bits,train_bit_density,best_val_loss,best_epoch
0,frozen_maccs_166_bce,maccs_166,166,0.265088,0.282593,24
1,frozen_map4_2048_bce,map4_2048,2048,0.267243,0.475535,21
2,frozen_morgan_2048_bce,morgan_2048,2048,0.022366,0.063586,32


Interpretation:
- Do NOT compare raw BCE losses across different fingerprint types.
- Different n_bits and bit densities change baseline BCE scale.
- Use val loss only for within-type comparisons (e.g., Morgan frozen vs Morgan fine-tuned).
- For cross-type comparison, use evaluation metrics: per-bit AUROC, Tanimoto@optimal-threshold, retrieval acc@k.


## Notes

- This notebook assumes you already have pre-computed frozen SSL embeddings in an HDF5 file with a 2D dataset of shape (N, 1024).
- If your embeddings live under a different path or key, set EMBEDDING_HDF5 and/or EMBEDDING_KEYS_CANDIDATES accordingly in Cell 2.
- Best checkpoint per condition is saved to: results/frozen_deepsets_baselines/<run_tag>/checkpoints/best.ckpt
- To keep this baseline CPU/M1-friendly, model size is tiny and there is no backbone forward pass.